# v6 Q2 runs (GPU 1): Hindi only + a push at every phase boundary, branched from phase 30
**Question (Q2):** is the self's stability special to *switching languages*, or would *any* big movement of the network give the same pattern?
* Branch: `v4b_only_hi_s0` phase 30 → phases 31, 32, ... all Hindi, same recipe.
* Change: at the start of every phase, add Gaussian noise `push × (each weight matrix's current std)` to every weight matrix, then train the phase as usual. So the network is pushed hard at every boundary but **never switches language**.
* Pick the push sizes so that the network moves about as much at a boundary as the continual run does at a switch (task-neuron change). Use `analysis/v6/quick_checks/summary.txt`: the noise row whose task change is closest to the continual switches' task change, and one a bit above it.
* Reading the result: at the same movement, if continual has a bigger relative gap than these runs, switching itself matters (H1). If they match, it is generic (H2/H3).

In [1]:
TRAIN_LANGS = ["hi"]

import os, sys, json, glob, subprocess, time
REPO = "/workspace/shreshth"
os.chdir(REPO); sys.path.insert(0, REPO)
from huggingface_hub import HfApi, hf_hub_download
api = HfApi(); HF_USER = api.whoami()["name"]; HF_REPO = HF_USER + "/cml-lang-self-checkpoints"
print("code files:", {f: os.path.exists(f) for f in ["common_v6.py", "train_v6.py", "tokenizer_v4"]})

# FLORES+ dev + devtest (training evaluates on devtest). Skips files already here.
codes = {"en": "eng_Latn", "hi": "hin_Deva", "zh": "cmn_Hans", "ar": "arb_Arab"}
os.makedirs("data/flores", exist_ok=True)
for l, code in codes.items():
    for split in ["dev", "devtest"]:
        p = f"data/flores/{split}.{l}.jsonl"
        if not os.path.exists(p):
            from datasets import load_dataset
            ds = load_dataset("openlanguagedata/flores_plus", code, split=split)
            with open(p, "w", encoding="utf-8") as f:
                for t in ds["text"]:
                    f.write(json.dumps({"text": t}, ensure_ascii=False) + "\n")
            print("downloaded", p, len(ds))

# training data: data/en-{hi,zh,ar}/{train,val}.jsonl
LANGS_NEEDED = TRAIN_LANGS
need = [f"en-{l}/train.jsonl" for l in LANGS_NEEDED] + [f"en-{l}/val.jsonl" for l in ["hi", "zh", "ar"]]   # val of all 3: forgetting is measured
missing = [f"data/{n}" for n in need if not os.path.exists(f"data/{n}")]
if missing:   # try the HF dataset backup (only exists if you uploaded it once, see push_data_v6.py)
    try:
        from huggingface_hub import snapshot_download
        snapshot_download(HF_USER + "/cml-lang-self-data", repo_type="dataset", local_dir="data",
                          allow_patterns=need)
    except Exception as e:
        print("no HF data backup:", type(e).__name__)
    missing = [m for m in missing if not os.path.exists(m)]
print("training data OK" if not missing else f"MISSING {missing}: copy data/ from the old machine or laptop (see push_data_v6.py)")

code files: {'common_v6.py': True, 'train_v6.py': True, 'tokenizer_v4': True}
training data OK


In [2]:
# look at the quick-check results to choose the push sizes
print(open("analysis/v6/quick_checks/summary.txt").read() if os.path.exists("analysis/v6/quick_checks/summary.txt") else "quick checks not finished")

      check                                label  task  self_ch  random   gap  self_size  stays  rel_gap_pct
      noise           continual p45 + noise 0.03  0.17     0.13    0.15  0.05       0.54   0.97        27.64
      noise            continual p45 + noise 0.1  1.92     1.40    1.63  0.53       0.53   0.93        27.45
      noise            continual p45 + noise 0.2  7.51     5.52    6.47  1.99       0.51   0.85        26.49
      noise            continual p45 + noise 0.4 27.98    21.33   25.37  6.64       0.38   0.61        23.75
      noise            continual p45 + noise 0.5 41.83    32.77   39.03  9.06       0.29   0.46        21.66
      noise            continual p45 + noise 0.6 55.94    45.25   53.75 10.69       0.18   0.28        19.11
      noise                only p45 + noise 0.03  0.23     0.18    0.21  0.05       0.47   0.98        21.39
      noise                 only p45 + noise 0.1  2.53     2.01    2.29  0.52       0.46   0.93        20.66
      noise        

In [3]:
# the checkpoint we branch from (skips if already here)
SRC_RUN, SRC_PHASE = "v4b_only_hi_s0", 30
have = glob.glob(f"runs/{SRC_RUN}/ckpt/p{SRC_PHASE:03d}_*.pt")
if not have:
    name = next(f for f in api.list_repo_files(HF_REPO) if f.startswith(f"{SRC_RUN}/ckpt/p{SRC_PHASE:03d}_"))
    hf_hub_download(HF_REPO, name, local_dir="runs")
    have = glob.glob(f"runs/{SRC_RUN}/ckpt/p{SRC_PHASE:03d}_*.pt")
INIT = have[0]; print("branch from:", INIT, f"{os.path.getsize(INIT) / 1e6:.0f} MB")

branch from: runs/v4b_only_hi_s0/ckpt/p030_hi.pt 267 MB


In [4]:
GPU = "1"
N_PHASES = 90
PUSH = [0.5, 0.6]                 # <- SET FROM THE QUICK CHECKS (x current std, once per phase boundary)
JOBS = {f"v6_hi_push{s:g}_from30": ["--mode", "only_hi", "--boundary_push", str(s)] for s in PUSH}
print(JOBS)

{'v6_hi_push0.5_from30': ['--mode', 'only_hi', '--boundary_push', '0.5'], 'v6_hi_push0.6_from30': ['--mode', 'only_hi', '--boundary_push', '0.6']}


In [5]:
def launch(run, args):
    os.makedirs("logs", exist_ok=True)
    log = open(f"logs/{run}.out", "a")
    cmd = [sys.executable, "-u", "train_v6.py", "--gpu", GPU, "--run", run, "--init", INIT,
           "--start_phase", str(SRC_PHASE), "--n_phases", str(N_PHASES)] + args
    p = subprocess.Popen(cmd, stdout=log, stderr=subprocess.STDOUT, start_new_session=True)  # survives the kernel
    print(f"started {run} (pid {p.pid}) -> logs/{run}.out"); return p

procs = {run: launch(run, a) for run, a in JOBS.items()}

started v6_hi_push0.5_from30 (pid 9009) -> logs/v6_hi_push0.5_from30.out
started v6_hi_push0.6_from30 (pid 9010) -> logs/v6_hi_push0.6_from30.out


In [7]:
# check progress any time (safe to rerun; does not touch the jobs)
for run in JOBS:
    print("=" * 30, run); os.system(f"tail -n 4 logs/{run}.out")
    f = f"runs/{run}/phases.jsonl"
    if os.path.exists(f):
        R = [json.loads(l) for l in open(f)]
        print("  phases done:", [r["phase"] for r in R][-6:], "| last chrF:", R[-1].get("chrf_masked"), "| health:", R[-1].get("health"))
os.system("nvidia-smi --query-gpu=index,memory.used,utilization.gpu --format=csv")

============================== v6_hi_push0.5_from30
    self.gen.throw(value)
  File "/venv/main/lib/python3.12/site-packages/httpx2/_transports/default.py", line 115, in map_httpcore_exceptions
    raise mapped_exc(message) from exc
httpx2.ConnectError: [SSL: UNEXPECTED_EOF_WHILE_READING] EOF occurred in violation of protocol (_ssl.c:1010)
============================== v6_hi_push0.6_from30
New Data Upload               : |          |  0.00B /  0.00B,  0.00B/s  
Validating                    : ██████████| 100%            
  ....6_from30/ckpt/p030_hi.pt: 100%|██████████|  267MB /  267MB            
  [p31 hi] step    500 | lr 2.0e-04 | train 3.591 | val 1.728 (best 1.728) | window Δ n/a | stable | 75k tok/s | phase 1m38s | run ETA: after first phase | health {'wnorm': 257.726, 'rank_enc1': 864.7, 'rank_dec1': 747.1, 'rarely_active': 0.0065}
  phases done: [30] | last chrF: {'hi': 49.58, 'zh': 0.58, 'ar': 1.78} | health: None
index, memory.used [MiB], utilization.gpu [%]
0, 21854 MiB, 8

0

In [ ]:
# to stop both jobs (only if needed). Rerunning the launch cell later RESUMES from the last saved phase.
# for run in JOBS: os.system(f"pkill -f 'train_v6.py.*--run {run}'")